In [2]:
!pip install -q pypdf

In [3]:
import os
print(os.getcwd())

files=os.listdir("/content")
policy_files=[ file for file in files if file.endswith((".txt",".pdf"))]
print(policy_files)


/content
['05_Health_Insurance_Policy.txt', '01_Vacation_Policy.txt', '06_Working_Hours_Policy.pdf', '07_Training_Policy.txt', '03_Remote_Work_Policy.txt', '02_Sick_Leave_Policy.pdf', '08_Travel_Policy.pdf', '04_Parental_Leave_Policy.pdf']


In [4]:
from pathlib import Path
from pypdf import PdfReader
print("pdf reader imported successfully")

documents=[]
for file_name in policy_files:
  file_path=Path("/content")/file_name

  if file_path.suffix==".txt":
    text=file_path.read_text(encoding="utf-8")
  elif file_path.suffix==".pdf":
    reader=PdfReader(file_path)
    text= ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text+= page_text + "\n"
  else:
    print(f"un supported file {file_name}")
    continue
  documents.append({"source":file_name,"text":text})
print(len(documents))



pdf reader imported successfully
8


In [5]:
# CHUNKING
!pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter


def create_chunks(documents,chunks_size,chunk_overlap):

  splitter=RecursiveCharacterTextSplitter(
      chunk_size=chunks_size,
      chunk_overlap=chunk_overlap
  )

  chunks=[]
  for doc in documents:
    text_chunks=splitter.split_text(doc["text"])
    for chunk in text_chunks:
      chunks.append({
        "source":doc["source"],"text":chunk
      })

  print(f"total chunks count of chunks_500 :{len(chunks)}")
  print(chunks[0])
  return chunks


# chunk_250=create_chunks(documents,250,50)
# print(chunk_250)

chunks_500=create_chunks(documents,500,100)
print(chunks_500)


total chunks count of chunks_500 :8
{'source': '05_Health_Insurance_Policy.txt', 'text': 'Health Insurance Policy\n\nSmartHR offers eligible employees health insurance that includes medical, dental, and vision coverage. Coverage details, provider networks, premiums, and enrollment dates are described in the plan documents. Employees should review the plan documents before selecting benefits. Questions about enrollment should be directed to Human Resources.'}
[{'source': '05_Health_Insurance_Policy.txt', 'text': 'Health Insurance Policy\n\nSmartHR offers eligible employees health insurance that includes medical, dental, and vision coverage. Coverage details, provider networks, premiums, and enrollment dates are described in the plan documents. Employees should review the plan documents before selecting benefits. Questions about enrollment should be directed to Human Resources.'}, {'source': '01_Vacation_Policy.txt', 'text': 'Vacation Policy\n\nSmartHR employees receive 20 vacation days 

In [6]:
!pip install -q langchain-core

In [7]:

from langchain_core.documents import Document
langchaindoc=[
    Document(
        page_content=chunk["text"],
        metadata={"source":chunk["source"],
                  "topic":chunk["source"].replace("_"," ").replace(".pdf"," ").replace(".txt"," ")}
    )

    for chunk in chunks_500
]

print("local documents :",len(langchaindoc))
print(langchaindoc[0])

local documents : 8
page_content='Health Insurance Policy

SmartHR offers eligible employees health insurance that includes medical, dental, and vision coverage. Coverage details, provider networks, premiums, and enrollment dates are described in the plan documents. Employees should review the plan documents before selecting benefits. Questions about enrollment should be directed to Human Resources.' metadata={'source': '05_Health_Insurance_Policy.txt', 'topic': '05 Health Insurance Policy '}


In [8]:
!pip install -q rank_bm25

from rank_bm25 import BM25Okapi

chunk_text=[doc.page_content for doc in langchaindoc]

corpus_tokens=[text.lower().split() for text in chunk_text]

bm_25=BM25Okapi(corpus_tokens)

print("bm 25 isready")



bm 25 isready


In [9]:
!pip install -q sentence-transformers

In [10]:
#EMBEDDING--LOADING MINI LLM
from sentence_transformers import SentenceTransformer
embedding_model=SentenceTransformer("sentence-transformers/all-MiniLM-L6-V2")
chunk_text=[chunk["text"] for chunk in chunks_500]
documents_embedding=embedding_model.encode(chunk_text)

print(f"number of embeddings : {len(documents_embedding)}")
print(f"Embeddings shape : {documents_embedding.shape}")
print(f"first 10 values: {documents_embedding[0][:10]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

number of embeddings : 8
Embeddings shape : (8, 384)
first 10 values: [-0.04654424  0.13035196  0.06192549 -0.02227777  0.05881516  0.08652259
  0.0462016   0.02525098 -0.08178341  0.06996854]


In [11]:
# CHROMA DB
!pip install -q chromadb
import chromadb
client=chromadb.Client()
print(f"chromadb created successfuly")

collection=client.get_or_create_collection(
    name="smart_hrpolicies"
)
print("collection created succesfully")


ids=[f"chunk {i}" for i in range(len(chunks_500))]

collection.upsert(
    ids=ids,
    documents=[doc.page_content for doc in  langchaindoc],
    embeddings=documents_embedding.tolist(),
    metadatas=[doc.metadata for doc in langchaindoc]
)
print(f"collection count {collection.count()}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
chromadb created successfuly
collection created succesfully
collection count 8


In [12]:
# Retrieval

question="How much anual leave do employees recieve?"

question_embedding=embedding_model.encode([question])

results=collection.query(question_embedding.tolist(),n_results=3)

print("results",results)

print(f"retrieved text: {results["documents"]}")
print(f"metadata: {results["metadatas"]}")
print(f"Distances: {results["distances"]}")


#RETRIEVING ONE MATCHING CHUNK
result_k1=collection.query(
    query_embeddings=question_embedding.tolist(),
    n_results=1
)

#RETRIEVING THREE MATCHING CHUNKS

result_k3=collection.query(
    query_embeddings=question_embedding.tolist(),
    n_results=3
)

print("K=1:", result_k1["documents"])
print("K=3:", result_k3["documents"])

# #Filtered by using metadata

filtered_results=collection.query(
    query_embeddings=question_embedding.tolist(),
    n_results=3,
    where={"source":"01_Vacation_Policy.txt"}


)

print("filtered text:", filtered_results["documents"])
print("filtered metadata:", filtered_results["metadatas"])
print("filtered distances:", filtered_results["distances"])




results {'ids': [['chunk 5', 'chunk 1', 'chunk 3']], 'embeddings': None, 'documents': [['Sick Leave Policy\nSmartHR employees receive 10 sick days per calendar year. Sick leave is intended for an employee’s illness,\nmedical appointments, or recovery. Employees should notify their manager as soon as reasonably possible\nwhen they need sick leave. A manager may request appropriate documentation when required by company\nprocedure.', 'Vacation Policy\n\nSmartHR employees receive 20 vacation days per calendar year. Vacation should be requested in advance through the company’s leave process. A manager must approve vacation requests before an employee takes the leave. Employees should check their remaining balance before planning extended time away.', 'Training and Development Policy\n\nEligible SmartHR employees have an annual training allowance of $1,000. The allowance may be used for approved job-related courses, certifications, or learning materials. Employees must obtain manager approv

In [13]:
for i, doc in enumerate(results["documents"][0], start=1):
    print(f"\nChunk {i}:")
    print(doc)


Chunk 1:
Sick Leave Policy
SmartHR employees receive 10 sick days per calendar year. Sick leave is intended for an employee’s illness,
medical appointments, or recovery. Employees should notify their manager as soon as reasonably possible
when they need sick leave. A manager may request appropriate documentation when required by company
procedure.

Chunk 2:
Vacation Policy

SmartHR employees receive 20 vacation days per calendar year. Vacation should be requested in advance through the company’s leave process. A manager must approve vacation requests before an employee takes the leave. Employees should check their remaining balance before planning extended time away.

Chunk 3:
Training and Development Policy

Eligible SmartHR employees have an annual training allowance of $1,000. The allowance may be used for approved job-related courses, certifications, or learning materials. Employees must obtain manager approval before making a purchase or registering for training. Reimbursement re

In [14]:

import transformers
import huggingface_hub

print("Transformers:", transformers.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)


Transformers: 5.20.2
Hugging Face Hub: 1.33.0


In [15]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("FLAN-T5 loaded successfully!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 loaded successfully!


In [22]:
from sentence_transformers import CrossEncoder

cross_encoder=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("cross encoder is ready")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

cross encoder is ready


In [16]:

# #
# results = collection.query(
#     query_embeddings=question_embedding.tolist(),
#     n_results=5
# )
# print(result.keys())

In [17]:
question = "How much annual leave do employees receive?"

question_embedding = embedding_model.encode([question])

result = collection.query(
    query_embeddings=question_embedding.tolist(),
    n_results=3
)

chromadb_docs=result["documents"][0]
for i, doc in enumerate(result["documents"][0]):
    print(f"\nChunk {i+1}:")
    print(doc)
    print("Source:", result["metadatas"][0][i]["source"])



tokenized_query=question.lower().split()
bm25_doc=bm_25.get_top_n(tokenized_query,chunk_text,n=1)
final_chunk=list(set(chromadb_docs+bm25_doc))

context = "\n".join(final_chunk)
prompt = f"""Answer the question using only the context below.
If the answer is not present, say you don't know.

Context:
{context}

Question:
{question}

Answer:"""

inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
outputs = model.generate(**inputs, max_new_tokens=50)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)


print("Question:", question)
print("Answer:", answer)


Chunk 1:
Sick Leave Policy
SmartHR employees receive 10 sick days per calendar year. Sick leave is intended for an employee’s illness,
medical appointments, or recovery. Employees should notify their manager as soon as reasonably possible
when they need sick leave. A manager may request appropriate documentation when required by company
procedure.
Source: 02_Sick_Leave_Policy.pdf

Chunk 2:
Vacation Policy

SmartHR employees receive 20 vacation days per calendar year. Vacation should be requested in advance through the company’s leave process. A manager must approve vacation requests before an employee takes the leave. Employees should check their remaining balance before planning extended time away.
Source: 01_Vacation_Policy.txt

Chunk 3:
Training and Development Policy

Eligible SmartHR employees have an annual training allowance of $1,000. The allowance may be used for approved job-related courses, certifications, or learning materials. Employees must obtain manager approval before

In [28]:
import gradio as gr
import numpy as np

def smarthr(message, history):
  current_message_embedding=embedding_model.encode([message])
  result = collection.query(
        query_embeddings=current_message_embedding.tolist(),
        n_results=3
    )
  # chromadb result
  chromadb_result=result["documents"][0]

  tokenized_query=message.lower().split()

  # bm_25 algorthm

  bm_25result=bm_25.get_top_n(tokenized_query, chunk_text, n=2)

  final_chunk=list(set(chromadb_result+bm_25result))

  pairs=[[message,doc] for doc in final_chunk]

  cross_Encoder_scores=cross_encoder.predict(pairs)

  best_index=np.argmax(cross_Encoder_scores)

  best_chunk=final_chunk[best_index]

  context=best_chunk

  prompt = f"""Answer the question concisely using ONLY the context below.
    If the exact answer is not clearly stated in the context, strictly output "I don't know". Do not guess.

Context:
{context}
Question:
{message}

Answer:"""

  inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

  outputs = model.generate(
        **inputs,
        max_new_tokens=50
    )

  answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

  return answer

demo = gr.ChatInterface(
    fn=smarthr,
    title="SmartHR Assistant",
    description="Ask questions about HR policies."
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/29 10:52:09 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>

In [27]:
import gradio as gr
def smarthr(message, history):
  current_message_embedding=embedding_model.encode([message])
  result = collection.query(
        query_embeddings=current_message_embedding.tolist(),
        n_results=1
    )
  chromadb_result=result["documents"][0]


  content="\n\n".join(chromadb_result)
  prompt = f"""Answer the question concisely using ONLY the context below.
    If the exact answer is not clearly stated in the context, strictly output "I don't know". Do not guess.

Context:
{context}
Question:
{message}

Answer:"""

  inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

  outputs = model.generate(
        **inputs,
        max_new_tokens=50
    )

  answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

  return answer

demo = gr.ChatInterface(
    fn=smarthr,
    title="SmartHR Assistant",
    description="Ask questions about HR policies."
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/29 10:49:30 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>